In [1]:
import os, copy, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
from google.colab import drive

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/AML_Dataset/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CONFIG_NAME = "dinov2_frozenFalse_weightsTrue"
CHECKPOINT_PATH = f"{CHECKPOINT_DIR}/{CONFIG_NAME}_checkpoint.pth"

Mounted at /content/drive


In [2]:
from google.colab import files
uploaded = files.upload()

import zipfile
with zipfile.ZipFile("WaRP-C-preprocessed.zip", "r") as z:
    z.extractall("/content/WaRP-C-preprocessed")

Saving WaRP-C-preprocessed.zip to WaRP-C-preprocessed.zip


In [3]:
PREPROCESSED_ROOT = "/content/WaRP-C-preprocessed"
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

CFG = {
    "num_classes": 28,
    "backbone_name": "dinov2_vitb14",
    "freeze_backbone":False,
    "use_class_weights": True,
    "lr": 3e-4,
    "min_lr": 1e-6,
    "weight_decay":0.05,
    "label_smoothing":0.1,
    "warmup_epochs": 3,
    "num_epochs":15,
    "blend_alpha": 0.4,
    "early_stop_patience":5,
}

full_train_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.2)),
])

eval_pipeline = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [4]:
def _make_flat_dataset(root_dir, transform):
    samples, class_to_idx = [], {}
    for superclass in sorted(os.listdir(root_dir)):
        sp = os.path.join(root_dir, superclass)
        if not os.path.isdir(sp): continue
        for subclass in sorted(os.listdir(sp)):
            scp = os.path.join(sp, subclass)
            if not os.path.isdir(scp): continue
            if subclass not in class_to_idx:
                class_to_idx[subclass] = len(class_to_idx)
            for img_name in os.listdir(scp):
                if img_name.lower().endswith(".jpg"):
                    samples.append((os.path.join(scp, img_name), class_to_idx[subclass]))
    dataset = datasets.ImageFolder(root_dir, transform=transform)
    dataset.samples = dataset.imgs = samples
    dataset.targets = [s[1] for s in samples]
    dataset.classes = list(class_to_idx.keys())
    dataset.class_to_idx = class_to_idx
    return dataset


def get_dataloaders(root=PREPROCESSED_ROOT, batch_size=32, num_workers=2, seed=42):
    torch.manual_seed(seed)
    train_ds = _make_flat_dataset(f"{root}/train", transform=full_train_pipeline)
    val_ds  = _make_flat_dataset(f"{root}/val",   transform=eval_pipeline)
    test_ds  = _make_flat_dataset(f"{root}/test",  transform=eval_pipeline)

    train_loader= DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                               num_workers=num_workers, pin_memory=True, drop_last=True)
    val_loader= DataLoader(val_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    test_loader= DataLoader(test_ds, batch_size=batch_size, shuffle=False,
                               num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader, train_ds, val_ds, test_ds


def get_class_counts(dataset):
    counts = np.bincount(dataset.targets, minlength=len(dataset.classes))
    return counts.astype(np.float32)


train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = get_dataloaders()
num_classes = len(train_loader.dataset.classes)
print(f"Classes: {num_classes} | Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Classes: 28 | Train: 7058 | Val: 1765 | Test: 1551


In [5]:
class DINOv2Classifier(nn.Module):
    def __init__(self, backbone, embed_dim, num_classes, freeze_backbone=False):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Linear(embed_dim, num_classes)
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)


backbone = torch.hub.load('facebookresearch/dinov2', CFG["backbone_name"])
embed_dim = backbone.embed_dim
model = DINOv2Classifier(backbone, embed_dim, num_classes, freeze_backbone=CFG["freeze_backbone"]).to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable:,}")

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vitb14/dinov2_vitb14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vitb14_pretrain.pth


100%|██████████| 330M/330M [00:01<00:00, 238MB/s]


Total params: 86,602,012 | Trainable: 86,602,012


In [6]:
#Ablation Study (fine-tuned+class-weighted)
if CFG["use_class_weights"]:
    samples_per_class = get_class_counts(train_ds)
    freq_inverse = 1.0 / (samples_per_class + 1e-6)
    loss_weights = torch.tensor(freq_inverse / freq_inverse.sum(), dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=loss_weights, label_smoothing=CFG["label_smoothing"])
else:
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

optimizer = optim.AdamW([
    {"params": model.backbone.parameters(), "lr": CFG["lr"] / 10},
    {"params": model.head.parameters(), "lr": CFG["lr"]},
], weight_decay=CFG["weight_decay"])

def get_lr_scale(epoch):
    if epoch < CFG["warmup_epochs"]:
        return (epoch + 1) / CFG["warmup_epochs"]
    decay_progress = (epoch - CFG["warmup_epochs"]) / max(1, CFG["num_epochs"] - CFG["warmup_epochs"])
    return CFG["min_lr"] / CFG["lr"] + 0.5 * (1 - CFG["min_lr"] / CFG["lr"]) * (1 + np.cos(np.pi * decay_progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, get_lr_scale)

In [7]:
def mixup_data(x, y, alpha=1.0):
    blend_ratio = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0)).to(x.device)
    blended_imgs = blend_ratio * x + (1 - blend_ratio) * x[index, :]
    return blended_imgs, y, y[index], blend_ratio

def mixup_criterion(criterion, pred, labels_orig, labels_mixed, blend_ratio):
    return blend_ratio * criterion(pred, labels_orig) + (1 - blend_ratio) * criterion(pred, labels_mixed)

def train_one_epoch(model, loader, optimizer, criterion, blend_alpha=0.0):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        if blend_alpha > 0:
            imgs, labels_orig, labels_mixed, blend_ratio = mixup_data(imgs, labels, blend_alpha)
        optimizer.zero_grad()
        logits = model(imgs)
        loss = mixup_criterion(criterion, logits, labels_orig, labels_mixed, blend_ratio) if blend_alpha > 0 else criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, eval_device=None):
    eval_device = eval_device or device
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for imgs, labels in loader:
        imgs, labels = imgs.to(eval_device), labels.to(eval_device)
        logits = model(imgs)
        loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return running_loss / total, correct / total, all_preds, all_labels

In [9]:
#Training
history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_acc, patience_counter, start_epoch = 0.0, 0, 0

if os.path.exists(CHECKPOINT_PATH):
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state"])
    optimizer.load_state_dict(ckpt["optimizer_state"])
    scheduler.load_state_dict(ckpt["scheduler_state"])
    start_epoch = ckpt["epoch"] + 1
    best_val_acc = ckpt["best_val_acc"]
    patience_counter = ckpt["patience_counter"]
    history = ckpt["history"]
    print(f"Resumed {CONFIG_NAME} from epoch {start_epoch} (best val acc so far: {best_val_acc:.4f})")
else:
    print(f"No checkpoint found for {CONFIG_NAME} ,starting fresh.")

best_state = copy.deepcopy(model.state_dict())

for epoch in range(start_epoch, CFG["num_epochs"]):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, blend_alpha=CFG["blend_alpha"])
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)
    scheduler.step()

    epoch_lr = optimizer.param_groups[0]["lr"]
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["train_acc"].append(train_acc)
    history["val_acc"].append(val_acc)

    print(f"Epoch [{epoch+1:02d}/{CFG['num_epochs']}] "
          f"Train Loss: {train_loss:.4f} Acc: {train_acc:.3f} | "
          f"Val Loss: {val_loss:.4f} Acc: {val_acc:.3f} | LR: {epoch_lr:.2e}")

    improved = val_acc > best_val_acc
    if improved:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
        print(f"New best val acc: {best_val_acc:.4f}")
    else:
        patience_counter += 1

    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "best_model_state": best_state,
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict(),
        "best_val_acc": best_val_acc,
        "patience_counter": patience_counter,
        "history": history,
        "cfg": CFG,
    }, CHECKPOINT_PATH)

    if patience_counter >= CFG["early_stop_patience"]:
        print("Early stopping triggered.")
        break

model.load_state_dict(best_state)
print(f"\nBest val accuracy: {best_val_acc:.4f}")

Resumed dinov2_frozenFalse_weightsTrue from epoch 15 (best val acc so far: 0.8210)

Best val accuracy: 0.8210


In [10]:
test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, criterion)
precision, recall, f1, _ = precision_recall_fscore_support(test_labels, test_preds, average="weighted", zero_division=0)

print(f"Accuracy : {test_acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall : {recall:.4f}")
print(f"F1-score: {f1:.4f}")

cm = confusion_matrix(test_labels, test_preds)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Accuracy : 0.8227
Precision: 0.8383
Recall : 0.8227
F1-score: 0.8241


In [11]:
final_results = {
    "config": {"freeze_backbone": CFG["freeze_backbone"], "use_class_weights": CFG["use_class_weights"]},
    "test_acc": float(test_acc),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1),
    "history": history,
}

with open(f"{CONFIG_NAME}_results.json", "w") as f:
    json.dump(final_results, f, indent=2)

torch.save(model.state_dict(), f"{CONFIG_NAME}_final.pth")

files.download(f"{CONFIG_NAME}_results.json")
files.download(f"{CONFIG_NAME}_final.pth")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [12]:
#Quantization
import torch.quantization

fp32_model_cpu = copy.deepcopy(model).to("cpu").eval()

quantized_model = torch.quantization.quantize_dynamic(
    fp32_model_cpu, {nn.Linear}, dtype=torch.qint8
)

def get_model_size_mb(m, filename="temp_model.pth"):
    torch.save(m.state_dict(), filename)
    size_mb = os.path.getsize(filename) / (1024 * 1024)
    os.remove(filename)
    return size_mb

fp32_size = get_model_size_mb(fp32_model_cpu)
int8_size = get_model_size_mb(quantized_model)

print(f"FP32 model size : {fp32_size:.2f} MB")
print(f"INT8 model size : {int8_size:.2f} MB")


/tmp/ipykernel_2491/3883583024.py:6: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  quantized_model = torch.quantization.quantize_dynamic(


FP32 model size : 330.42 MB
INT8 model size : 87.40 MB


In [13]:
import copy

# Accuracy comparison
cpu = torch.device("cpu")

# Move a copy of the criterion to CPU for evaluation on CPU models
criterion_cpu = copy.deepcopy(criterion).to(cpu)

fp32_loss, fp32_acc, fp32_preds, fp32_labels = evaluate(fp32_model_cpu, test_loader, criterion_cpu, eval_device=cpu)
fp32_prec, fp32_rec, fp32_f1, _ = precision_recall_fscore_support(
    fp32_labels, fp32_preds, average="weighted", zero_division=0
)

int8_loss, int8_acc, int8_preds, int8_labels = evaluate(quantized_model, test_loader, criterion_cpu, eval_device=cpu)
int8_prec, int8_rec, int8_f1, _ = precision_recall_fscore_support(
    int8_labels, int8_preds, average="weighted", zero_division=0
)

print(f"FP32 (CPU) - Acc: {fp32_acc:.4f} | Prec: {fp32_prec:.4f} | Rec: {fp32_rec:.4f} | F1: {fp32_f1:.4f}")
print(f"INT8 (CPU) - Acc: {int8_acc:.4f} | Prec: {int8_prec:.4f} | Rec: {int8_rec:.4f} | F1: {int8_f1:.4f}")

fp32_cm = confusion_matrix(fp32_labels, fp32_preds)
int8_cm = confusion_matrix(int8_labels, int8_preds)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


FP32 (CPU) - Acc: 0.8227 | Prec: 0.8383 | Rec: 0.8227 | F1: 0.8241
INT8 (CPU) - Acc: 0.7002 | Prec: 0.7568 | Rec: 0.7002 | F1: 0.6954


In [14]:
def measure_latency(m, input_size=(1, 3, 224, 224), n_runs=50, warmup=5):
    m.eval()
    dummy_input = torch.randn(input_size)
    with torch.no_grad():
        for _ in range(warmup):
            _ = m(dummy_input)
        start = time.time()
        for _ in range(n_runs):
            _ = m(dummy_input)
        elapsed = time.time() - start
    return (elapsed / n_runs) * 1000

fp32_latency_ms = measure_latency(fp32_model_cpu)
int8_latency_ms = measure_latency(quantized_model)

print(f"FP32 CPU latency : {fp32_latency_ms:.2f} ms/image")
print(f"INT8 CPU latency : {int8_latency_ms:.2f} ms/image")


FP32 CPU latency : 1051.89 ms/image
INT8 CPU latency : 652.95 ms/image


In [15]:
quant_results = {
    "fp32_size_mb": fp32_size, "int8_size_mb": int8_size,
    "compression_ratio": fp32_size / int8_size,
    "fp32_acc": fp32_acc, "int8_acc": int8_acc,
    "fp32_precision": fp32_prec, "int8_precision": int8_prec,
    "fp32_recall": fp32_rec, "int8_recall": int8_rec,
    "fp32_f1": fp32_f1, "int8_f1": int8_f1,
    "fp32_latency_ms": fp32_latency_ms, "int8_latency_ms": int8_latency_ms,
    "speedup": fp32_latency_ms / int8_latency_ms,
}

with open("dinov2_quantization_results.json", "w") as f:
    json.dump(quant_results, f, indent=2)

torch.save(quantized_model.state_dict(), "dinov2_int8.pth")

files.download("dinov2_quantization_results.json")
files.download("dinov2_int8.pth")



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>